# 03. Model Evaluation

02_model_training에서 준비한 평균 특징 데이터를 이용하여
추가 모델 비교, 입력 시간 비교, SHAP 분석,
최종 Test 평가, 시간순서 스트레스 테스트를 수행한다.

### 최종 기준
- 평균 특징 17개
- 축압기 90/100/115/130 기준 Stratified 70/15/15
- 네 부품 + stable_flag 동일 cycle_id
- 최종 Test는 모델 선택 후 한 번만 사용

## 1. 실행 전 데이터 준비

### 1-1. 라이브러리

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)



### 1-2. XGBoost 사용 가능 여부

In [ ]:
try:
    from xgboost import (
        XGBClassifier
    )

    XGBOOST_AVAILABLE = True

except Exception:
    XGBOOST_AVAILABLE = False

print(
    "XGBoost 사용 가능:",
    XGBOOST_AVAILABLE
)

### 1-3. SHAP 사용 가능 여부

In [ ]:
try:
    import shap

    SHAP_AVAILABLE = True

except Exception:
    SHAP_AVAILABLE = False

print(
    "SHAP 사용 가능:",
    SHAP_AVAILABLE
)

### 1-4. 데이터 경로

In [ ]:
processed_dir = Path(
    "../data/processed"
)

profile_path = Path(
    "../data/raw/uci_hydraulic/extracted/profile.txt"
)

### 1-5. 특징 데이터 불러오기

In [ ]:
features_10s = pd.read_parquet(
    processed_dir
    / "features_10s.parquet"
)

features_20s = pd.read_parquet(
    processed_dir
    / "features_20s.parquet"
)

features_30s = pd.read_parquet(
    processed_dir
    / "features_30s.parquet"
)

features_60s = pd.read_parquet(
    processed_dir
    / "features_60s.parquet"
)

### 1-6. profile 불러오기

In [ ]:
profile = pd.read_csv(
    profile_path,
    sep=r"\s+",
    header=None
)

profile.columns = [
    "cooler",
    "valve",
    "pump",
    "accumulator",
    "stable_flag"
]

profile.insert(
    0,
    "cycle_id",
    range(1, len(profile) + 1)
)

print(
    "profile:",
    profile.shape
)

### 1-7. 센서와 특징 목록

In [ ]:
sensor_names = [
    "PS1", "PS2", "PS3", "PS4", "PS5", "PS6",
    "EPS1",
    "FS1", "FS2",
    "TS1", "TS2", "TS3", "TS4",
    "VS1",
    "CE", "CP", "SE"
]

sampling_rates = {
    "PS1": 100, "PS2": 100, "PS3": 100,
    "PS4": 100, "PS5": 100, "PS6": 100,
    "EPS1": 100,
    "FS1": 10, "FS2": 10,
    "TS1": 1, "TS2": 1, "TS3": 1, "TS4": 1,
    "VS1": 1,
    "CE": 1, "CP": 1, "SE": 1
}

mean_feature_cols = [
    f"{sensor}_mean"
    for sensor in sensor_names
]

component_order = [
    "cooler",
    "valve",
    "pump",
    "accumulator"
]

# component_order는 JSON의 components 안에 들어갈 4개 부품만 의미한다.
# stable_flag는 별도의 예측 타깃이므로 target_order에 추가한다.
target_order = component_order + ["stable_flag"]

## 2. 특징 데이터 검사

### 2-1. 시간별 특징 묶기

In [ ]:
feature_frames = {
    "10s": features_10s,
    "20s": features_20s,
    "30s": features_30s,
    "60s": features_60s
}

expected_columns = (
    ["cycle_id"]
    + mean_feature_cols
)

### 2-2. 평균 특징 17개 확인

In [ ]:
for name, features in (
    feature_frames.items()
):
    if (
        features.columns.tolist()
        != expected_columns
    ):
        raise ValueError(
            f"{name} 특징 컬럼 오류"
        )

    print(
        name,
        "→",
        features.shape
    )

assert len(
    mean_feature_cols
) == 17

### 2-3. cycle_id와 결측값 확인

In [ ]:
for name, features in (
    feature_frames.items()
):
    assert not (
        features["cycle_id"]
        .duplicated()
        .any()
    )

    assert not (
        features[
            mean_feature_cols
        ]
        .isna()
        .any()
        .any()
    )

print(
    "특징 데이터 검사 완료"
)

## 3. 축압기 기준 Stratified 분할

### 3-1. 분할 함수

In [ ]:
def make_accumulator_stratified_split(
    profile,
    random_state=42
):
    # 전체의 15%를 Test로 분리
    dev_ids, test_ids = train_test_split(
        profile["cycle_id"],
        test_size=0.15,
        random_state=random_state,
        stratify=profile["accumulator"]
    )

    # 남은 85% 중 전체의 15%를 Validation으로 분리
    dev_profile = profile[
        profile["cycle_id"].isin(dev_ids)
    ].copy()

    train_ids, val_ids = train_test_split(
        dev_profile["cycle_id"],
        test_size=0.15 / 0.85,
        random_state=random_state,
        stratify=dev_profile["accumulator"]
    )

    return (
        sorted(map(int, train_ids)),
        sorted(map(int, val_ids)),
        sorted(map(int, test_ids))
    )

### 3-2. 분할 파일 경로

In [ ]:
split_path = (
    processed_dir
    / "split_ids_accumulator_stratified.json"
)

print(
    "분할 파일 존재:",
    split_path.exists()
)

### 3-3. 공통 분할 ID 불러오기

In [ ]:
if split_path.exists():
    with open(
        split_path,
        "r",
        encoding="utf-8"
    ) as file:
        split_data = json.load(
            file
        )

    train_ids = list(
        map(
            int,
            split_data[
                "train_ids"
            ]
        )
    )

    val_ids = list(
        map(
            int,
            split_data[
                "val_ids"
            ]
        )
    )

    test_ids = list(
        map(
            int,
            split_data[
                "test_ids"
            ]
        )
    )

else:
    train_ids, val_ids, test_ids = (
        make_accumulator_stratified_split(
            profile,
            random_state=42
        )
    )

### 3-4. 분할 개수 확인

In [ ]:
print(
    "Train:",
    len(train_ids)
)

print(
    "Validation:",
    len(val_ids)
)

print(
    "Test:",
    len(test_ids)
)

assert len(train_ids) == 1543
assert len(val_ids) == 331
assert len(test_ids) == 331

### 3-5. 분할 중복 검사

In [ ]:
assert set(
    train_ids
).isdisjoint(
    val_ids
)

assert set(
    train_ids
).isdisjoint(
    test_ids
)

assert set(
    val_ids
).isdisjoint(
    test_ids
)

print(
    "분할 중복 0개"
)

### 3-6. X / y 생성 함수

In [ ]:
def get_xy(
    features,
    ids,
    component
):
    feature_part = (
        features[
            features["cycle_id"].isin(ids)
        ]
        .sort_values("cycle_id")
        .reset_index(drop=True)
    )

    label_part = (
        profile[
            profile["cycle_id"].isin(ids)
        ]
        .sort_values("cycle_id")
        .reset_index(drop=True)
    )

    if (
        feature_part["cycle_id"].tolist()
        != label_part["cycle_id"].tolist()
    ):
        raise ValueError(
            "특징 데이터와 라벨의 cycle_id 순서가 다릅니다."
        )

    X = feature_part[mean_feature_cols].copy()
    y = label_part[component].copy()

    return X, y

## 3-7. 최종 예측 타깃

모델 입력 X는 기존과 동일하게 **센서 평균 17개**만 사용한다.

예측 y는 총 5개다.

- `cooler`
- `valve`
- `pump`
- `accumulator`
- `stable_flag`

`stable_flag`는 입력 특징이 아니라 **별도 이진분류 예측 결과(0/1)**다.

In [ ]:
print("부품 예측 대상 :", component_order)
print("전체 예측 대상 :", target_order)

assert target_order == [
    "cooler",
    "valve",
    "pump",
    "accumulator",
    "stable_flag"
]

## 4. 추가 모델 비교

### 4-1. RandomForest 10초 모델

In [ ]:
rf_compare_results = []
rf_compare_models = {}

for component in target_order:
    X_train, y_train = get_xy(
        features_10s,
        train_ids,
        component
    )

    X_val, y_val = get_xy(
        features_10s,
        val_ids,
        component
    )

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_val
    )

    rf_compare_models[
        component
    ] = model

    rf_compare_results.append({
        "component":
            component,
        "model":
            "RandomForest",
        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),
        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            )
    })

rf_compare_results = pd.DataFrame(
    rf_compare_results
)

display(
    rf_compare_results
)

### 4-3. ExtraTrees 10초 모델

In [ ]:
extra_compare_results = []
extra_compare_models = {}

for component in target_order:
    X_train, y_train = get_xy(
        features_10s,
        train_ids,
        component
    )

    X_val, y_val = get_xy(
        features_10s,
        val_ids,
        component
    )

    model = ExtraTreesClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_val
    )

    extra_compare_models[
        component
    ] = model

    extra_compare_results.append({
        "component":
            component,
        "model":
            "ExtraTrees",
        "accuracy":
            accuracy_score(
                y_val,
                pred
            ),
        "macro_f1":
            f1_score(
                y_val,
                pred,
                average="macro",
                zero_division=0
            )
    })

extra_compare_results = pd.DataFrame(
    extra_compare_results
)

display(
    extra_compare_results
)

### 4-4. XGBoost 10초 모델

In [ ]:
xgb_compare_results = []
xgb_compare_models = {}

if XGBOOST_AVAILABLE:
    for component in target_order:
        X_train, y_train = get_xy(
            features_10s,
            train_ids,
            component
        )

        X_val, y_val = get_xy(
            features_10s,
            val_ids,
            component
        )

        encoder = LabelEncoder()

        y_train_encoded = (
            encoder.fit_transform(
                y_train
            )
        )

        model = XGBClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
            eval_metric="mlogloss"
        )

        model.fit(
            X_train,
            y_train_encoded
        )

        pred_encoded = (
            model.predict(
                X_val
            )
            .astype(int)
        )

        pred = (
            encoder.inverse_transform(
                pred_encoded
            )
        )

        xgb_compare_models[
            component
        ] = (
            model,
            encoder
        )

        xgb_compare_results.append({
            "component":
                component,
            "model":
                "XGBoost",
            "accuracy":
                accuracy_score(
                    y_val,
                    pred
                ),
            "macro_f1":
                f1_score(
                    y_val,
                    pred,
                    average="macro",
                    zero_division=0
                )
        })

    xgb_compare_results = pd.DataFrame(
        xgb_compare_results
    )

    display(
        xgb_compare_results
    )

else:
    print(
        "XGBoost가 설치되지 않아 건너뜁니다."
    )

### 4-5. 모델 비교 결과 통합

In [ ]:
compare_frames = [
    rf_compare_results,
    extra_compare_results
]

if XGBOOST_AVAILABLE:
    compare_frames.append(
        xgb_compare_results
    )

model_compare_results = pd.concat(
    compare_frames,
    ignore_index=True
)

display(
    model_compare_results
)

### 4-6. 모델별 평균 성능

In [ ]:
model_compare_summary = (
    model_compare_results
    .groupby(
        "model"
    )[
        [
            "accuracy",
            "macro_f1"
        ]
    ]
    .mean()
    .sort_values(
        "macro_f1",
        ascending=False
    )
)

display(
    model_compare_summary
)

## 4-7. 추가 모델 비교 결과 해석

RandomForest, ExtraTrees, XGBoost를 동일한
**축압기 기준 공통 Train/Validation cycle_id**에서 비교한다.

기존 출력값은 분할과 특징 정책이 변경되었으므로 그대로 사용하지 않고,
이 노트북을 다시 실행하여 새 결과를 확인한다.

In [ ]:
model_compare_summary = (
    model_compare_results
    .groupby("model")[
        ["accuracy", "macro_f1"]
    ]
    .mean()
    .sort_values(
        "macro_f1",
        ascending=False
    )
)

display(model_compare_summary)

selected_model_name = (
    model_compare_summary
    .index[0]
)

print(
    "Validation 평균 Macro F1 기준 1위 모델 :",
    selected_model_name
)

## 5. 입력 시간 비교

### 5-1. 시간별 특징 준비

In [ ]:
window_features = {
    10: features_10s,
    20: features_20s,
    30: features_30s,
    60: features_60s
}

### 5-2. RandomForest 시간별 학습

In [ ]:
rf_window_results = []
rf_window_models = {}

for seconds, features in (
    window_features.items()
):
    rf_window_models[
        seconds
    ] = {}

    for component in target_order:
        X_train, y_train = get_xy(
            features,
            train_ids,
            component
        )

        X_val, y_val = get_xy(
            features,
            val_ids,
            component
        )

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train
        )

        pred = model.predict(
            X_val
        )

        rf_window_models[
            seconds
        ][
            component
        ] = model

        rf_window_results.append({
            "window_sec":
                seconds,
            "component":
                component,
            "accuracy":
                accuracy_score(
                    y_val,
                    pred
                ),
            "macro_f1":
                f1_score(
                    y_val,
                    pred,
                    average="macro",
                    zero_division=0
                )
        })

### 5-3. 시간별 부품 성능

In [ ]:
rf_window_results = pd.DataFrame(
    rf_window_results
)

display(
    rf_window_results
)

### 5-4. 시간별 평균 성능

In [ ]:
window_summary = (
    rf_window_results
    .groupby(
        "window_sec"
    )[
        [
            "accuracy",
            "macro_f1"
        ]
    ]
    .mean()
)

display(
    window_summary
)

## 6. 최종 입력 시간 선정

### 6-1. 10초와 60초 성능 비교

In [ ]:
f1_10 = (
    window_summary
    .loc[
        10,
        "macro_f1"
    ]
)

f1_60 = (
    window_summary
    .loc[
        60,
        "macro_f1"
    ]
)

performance_ratio = (
    f1_10
    / f1_60
    * 100
    if f1_60 > 0
    else 0.0
)

print(
    f"10초 Macro F1: {f1_10:.6f}"
)

print(
    f"60초 Macro F1: {f1_60:.6f}"
)

print(
    f"성능 유지율: {performance_ratio:.2f}%"
)

### 6-2. 최종 입력 시간 결정

In [ ]:
final_window_sec = 10

print(
    "최종 입력 시간:",
    final_window_sec,
    "초"
)

## 6-3. 입력 시간 비교 해석

10초 / 20초 / 30초 / 60초 성능을 동일한 공통 분할에서 비교한다.

10초 입력의 평균 Macro F1은 0.989081로,
60초 입력의 0.977638 대비 약 101.17%의 성능을 유지하였다.

따라서 더 짧은 관측 시간으로도 충분한 성능을 확보할 수 있어
최종 입력 시간을 10초로 선정하였다.

## 7. SHAP 분석

### 7-1. SHAP 분석 준비

In [ ]:
final_validation_features = (
    window_features[
        final_window_sec
    ]
)

shap_top_features = {}

### 7-2. 예측 대상별 SHAP 상위 특징

In [ ]:
if SHAP_AVAILABLE:
    for component in target_order:
        model = (
            rf_window_models[
                final_window_sec
            ][
                component
            ]
        )

        X_val, y_val = get_xy(
            final_validation_features,
            val_ids,
            component
        )

        explainer = (
            shap.TreeExplainer(
                model
            )
        )

        shap_values = (
            explainer.shap_values(
                X_val
            )
        )

        n_features = (
            X_val.shape[1]
        )

        if isinstance(
            shap_values,
            list
        ):
            stacked = np.stack(
                [
                    np.abs(
                        np.asarray(values)
                    )
                    for values
                    in shap_values
                ],
                axis=0
            )

            mean_abs_shap = (
                stacked.mean(
                    axis=(0, 1)
                )
            )

        else:
            shap_array = np.abs(
                np.asarray(
                    shap_values
                )
            )

            if shap_array.ndim == 2:
                mean_abs_shap = (
                    shap_array.mean(
                        axis=0
                    )
                )

            elif shap_array.ndim == 3:
                feature_axes = [
                    axis
                    for axis, size
                    in enumerate(
                        shap_array.shape
                    )
                    if size == n_features
                ]

                if len(
                    feature_axes
                ) != 1:
                    raise ValueError(
                        "SHAP 특징 축 판별 실패"
                    )

                feature_axis = (
                    feature_axes[0]
                )

                reduce_axes = tuple(
                    axis
                    for axis
                    in range(
                        shap_array.ndim
                    )
                    if axis
                    != feature_axis
                )

                mean_abs_shap = (
                    shap_array.mean(
                        axis=reduce_axes
                    )
                )

            else:
                raise ValueError(
                    "예상하지 못한 SHAP 차원"
                )

        importance_df = pd.DataFrame({
            "feature":
                mean_feature_cols,
            "mean_abs_shap":
                mean_abs_shap
        }).sort_values(
            "mean_abs_shap",
            ascending=False
        )

        shap_top_features[
            component
        ] = (
            importance_df
            .head(10)
            .reset_index(drop=True)
        )

        print(
            f"\n===== {component.upper()} ====="
        )

        display(
            shap_top_features[
                component
            ]
        )

else:
    print(
        "SHAP가 설치되지 않아 건너뜁니다."
    )

### 7-3. SHAP 상위 특징을 센서 단위로 확인

In [ ]:
if SHAP_AVAILABLE:
    for component in target_order:
        top_df = (
            shap_top_features[component]
            .copy()
        )

        top_df["sensor"] = (
            top_df["feature"]
            .str.replace(
                "_mean",
                "",
                regex=False
            )
        )

        print(
            f"\n[{component}]"
        )

        display(
            top_df[
                [
                    "feature",
                    "sensor",
                    "mean_abs_shap"
                ]
            ]
        )

### 7-4. SHAP 해석 시 주의점

이번 최종 모델은 평균 특징만 사용하므로,
이전 119개 특징 실험에서 등장했던 `PS1_slope`, `FS2_rms`, `PS1_std` 등은
**현재 최종 SHAP 입력에는 포함되지 않는다.**

SHAP 결과는 현재 모델이 사용한 17개 `*_mean` 특징 안에서만 해석한다.

## 8. 최종 Test 평가

### 8-1. Train + Validation 결합

모델과 입력 시간 선택이 끝난 뒤에만 최종 Test를 사용한다.

In [ ]:
final_features = (
    window_features[
        final_window_sec
    ]
)

train_val_ids = sorted(
    set(train_ids)
    | set(val_ids)
)

### 8-2. 최종 모델 학습

In [ ]:
final_models = {}
final_test_results = []
final_confusion_matrices = {}

for component in target_order:
    X_train_val, y_train_val = get_xy(
        final_features,
        train_val_ids,
        component
    )

    X_test, y_test = get_xy(
        final_features,
        test_ids,
        component
    )

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_val,
        y_train_val
    )

    pred = model.predict(
        X_test
    )

    labels = sorted(
        profile[
            component
        ].unique()
    )

    accuracy = accuracy_score(
        y_test,
        pred
    )

    macro_f1 = f1_score(
        y_test,
        pred,
        labels=labels,
        average="macro",
        zero_division=0
    )

    cm = confusion_matrix(
        y_test,
        pred,
        labels=labels
    )

    final_models[
        component
    ] = model

    final_test_results.append({
        "component":
            component,
        "window_sec":
            final_window_sec,
        "accuracy":
            accuracy,
        "macro_f1":
            macro_f1,
        "test_count":
            len(y_test)
    })

    final_confusion_matrices[
        component
    ] = pd.DataFrame(
        cm,
        index=[
            f"actual_{label}"
            for label in labels
        ],
        columns=[
            f"pred_{label}"
            for label in labels
        ]
    )

### 8-2-1. RandomForest 통합모델 1개 파일로 저장

최종 학습된 5개 RandomForest 분류기를 하나의 `joblib` 번들에 묶어 저장한다.

저장되는 모델 파일은 **1개만** 생성한다.

```text
C:\ai-first-project\models\predict\integrated_rf.joblib
```

번들 내부에는 다음 정보가 함께 들어간다.

- `stable_flag`, `cooler`, `valve`, `pump`, `accumulator` 모델
- 평균 특징 17개 컬럼 순서
- 최종 입력 시간(`final_window_sec`)
- 각 타깃의 원래 클래스 값

별도의 `cooler.joblib`, `valve.joblib` 같은 파일은 생성하지 않는다.

In [ ]:
model_dir = Path(
    r"C:\ai-first-project\models\predict"
)

model_dir.mkdir(
    parents=True,
    exist_ok=True
)

integrated_model_path = (
    model_dir
    / "integrated_rf.joblib"
)

class_labels = {
    target: [
        int(value)
        for value in sorted(
            profile[target].unique()
        )
    ]
    for target in target_order
}

for name, model in final_models.items():
    print(
        name,
        type(model).__name__
    )

integrated_rf = {
    "model_type": "RandomForest",
    "models": final_models,
    "feature_names": list(
        mean_feature_cols
    ),
    "window_sec": int(
        final_window_sec
    ),
    "target_order": list(
        target_order
    ),
    "component_order": list(
        component_order
    ),
    "class_labels": class_labels,
    "split_policy": {
        "basis": "accumulator",
        "method": "Stratified 70/15/15",
        "random_state": 42
    }
}

joblib.dump(
    integrated_rf,
    integrated_model_path
)

print(
    "통합모델 저장 완료:"
)
print(
    integrated_model_path
)

# models/predict 폴더에서 생성된 모델 파일 확인
model_files = list(
    model_dir.glob("*.joblib")
)

print(
    "현재 joblib 파일:",
    [p.name for p in model_files]
)

### 8-3. 최종 Test 성능

In [ ]:
final_test_results = pd.DataFrame(
    final_test_results
)

display(
    final_test_results
)

print(
    "최종 Test 평균 Accuracy:",
    final_test_results[
        "accuracy"
    ].mean()
)

print(
    "최종 Test 평균 Macro F1:",
    final_test_results[
        "macro_f1"
    ].mean()
)

### 8-4. 최종 Test 혼동행렬

In [ ]:
for component in target_order:
    print(
        f"\n===== {component.upper()} ====="
    )

    display(
        final_confusion_matrices[
            component
        ]
    )

### 8-5. 최종 Test 해석

최종 Test는 모델/입력 시간 선택이 모두 끝난 뒤 한 번만 사용한다.
네 부품과 stable_flag 모두 동일한 축압기 기준 Stratified Test `cycle_id` 331개로 평가한다.

## 8-6. 최종 예측 JSON 형식

최종 `predict()` 결과는 아래 구조로 반환한다.

- `cycle_id` 출력 안 함
- `stable_state` 출력 안 함
- `confidence` 출력 안 함
- `stable_flag`는 모델이 예측한 0/1 값
- 네 부품 예측값은 `components` 안에 저장

예측 시에는 `integrated_rf.joblib` **한 파일만 로드**한다.

In [ ]:
def predict_result(
    feature_row,
    model_bundle=None
):
    """
    평균 특징 17개 1행을 입력받아
    통합 RandomForest 번들의 5개 모델로 예측하고
    최종 JSON 구조를 반환한다.
    """
    if model_bundle is None:
        model_bundle = integrated_rf

    if len(feature_row) != 1:
        raise ValueError(
            "predict_result에는 1개 샘플만 전달해야 합니다."
        )

    feature_names = model_bundle[
        "feature_names"
    ]

    models = model_bundle[
        "models"
    ]

    X_input = feature_row[
        feature_names
    ].copy()

    stable_pred = int(
        models["stable_flag"]
        .predict(X_input)[0]
    )

    component_predictions = {}

    for component in model_bundle[
        "component_order"
    ]:
        component_predictions[
            component
        ] = int(
            models[component]
            .predict(X_input)[0]
        )

    return {
        "stable_flag": stable_pred,
        "components":
            component_predictions
    }

### 8-7. Test 샘플 1개의 실제 예측 출력 확인

In [ ]:
# 저장된 RandomForest 통합모델 1개 파일을 다시 불러와 실제 예측 확인
loaded_rf = joblib.load(
    integrated_model_path
)

sample_test = (
    final_features[
        final_features[
            "cycle_id"
        ].isin(test_ids)
    ]
    .sort_values(
        "cycle_id"
    )
    .iloc[[0]]
)

prediction_json = predict_result(
    sample_test,
    model_bundle=loaded_rf
)

print(
    json.dumps(
        prediction_json,
        ensure_ascii=False,
        indent=2
    )
)

실행 결과 형식은 다음과 같다.

```json
{
  "stable_flag": 1,
  "components": {
    "cooler": 3,
    "valve": 100,
    "pump": 0,
    "accumulator": 130
  }
}
```

위 숫자는 예시이며, 실제 값은 해당 센서 평균 17개를 입력한 모델 예측 결과에 따라 달라진다.

## 9. 축압기 Test 분포 확인

### 9-1. 90 / 100 / 115 / 130 포함 여부

In [ ]:
accumulator_test_distribution = (
    profile.loc[
        profile[
            "cycle_id"
        ].isin(test_ids),
        "accumulator"
    ]
    .value_counts()
    .sort_index()
)

print(
    accumulator_test_distribution
)

assert set(
    accumulator_test_distribution.index
) == {
    90,
    100,
    115,
    130
}

assert (
    accumulator_test_distribution.min()
    > 0
)

print(
    "Test에 축압기 4개 클래스 모두 존재"
)

## 9-2. Stratified 최종 Test와 시간순 평가의 역할 구분

- **Stratified Test**: 축압기 `90/100/115/130` 클래스 비율을 유지한 메인 최종 성능
- **시간순서 Test**: 뒤쪽 구간 분포 변화에 대한 별도 일반화/강건성 시험

두 결과를 같은 의미의 Test 성능으로 섞어 해석하지 않는다.

## 10. 시간순서 일반화 스트레스 테스트

이 부분은 메인 분할이나 최종 성능 선정에 사용하지 않는다.
뒤쪽 시간 구간에서 분포 변화가 생겼을 때 성능이 얼마나 떨어지는지
추가 확인하기 위한 별도 시험이다.

### 10-1. 시간순서 분할

In [ ]:
time_train_ids = list(
    range(1, 1544)
)

time_val_ids = list(
    range(1544, 1875)
)

time_test_ids = list(
    range(1875, 2206)
)

### 10-2. 시간순서 모델 학습·평가

In [ ]:
stress_results = []

for component in target_order:
    X_time_train, y_time_train = get_xy(
        final_features,
        time_train_ids,
        component
    )

    X_time_test, y_time_test = get_xy(
        final_features,
        time_test_ids,
        component
    )

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_time_train,
        y_time_train
    )

    pred = model.predict(
        X_time_test
    )

    all_labels = sorted(
        profile[
            component
        ].unique()
    )

    stress_results.append({
        "component":
            component,
        "accuracy":
            accuracy_score(
                y_time_test,
                pred
            ),
        "macro_f1_all_classes":
            f1_score(
                y_time_test,
                pred,
                labels=all_labels,
                average="macro",
                zero_division=0
            ),
        "test_labels_present":
            sorted(
                y_time_test.unique()
            )
    })

### 10-3. 시간순서 스트레스 테스트 결과

In [ ]:
stress_results = pd.DataFrame(
    stress_results
)

display(
    stress_results
)

### 10-4. 시간순서 검증 결과 해석

Stratified 환경에서는 축압기 각 상태가 Train/Validation/Test에 고르게 포함되지만,
시간순서 뒤쪽 구간에는 특정 클래스가 없거나 분포가 크게 달라질 수 있다.

따라서 시간순서 성능 저하는 단순 모델 실패만을 의미하지 않고,
시간에 따른 센서 분포 변화와 클래스 구성 변화에 대한 모델의 한계를 보여주는
**스트레스 테스트 결과**로 기록한다.

## 11. 최종 무결성 검사

### 11-1. 분할과 특징 수 확인

In [ ]:
assert len(train_ids) == 1543
assert len(val_ids) == 331
assert len(test_ids) == 331

assert len(
    mean_feature_cols
) == 17

assert set(
    train_ids
).isdisjoint(
    test_ids
)

assert set(
    val_ids
).isdisjoint(
    test_ids
)

assert target_order == [
    "cooler",
    "valve",
    "pump",
    "accumulator",
    "stable_flag"
]

print(
    "분할, 특징 수, 예측 타깃 검사 통과"
)

### 11-2. 모든 특징 파일 컬럼 확인

In [ ]:
for features in [
    features_10s,
    features_20s,
    features_30s,
    features_60s
]:
    assert (
        features.columns.tolist()
        == (
            ["cycle_id"]
            + mean_feature_cols
        )
    )

print(
    "03_model_evaluation 최종 검사 통과"
)